# Сергушов Павел ПМ22-4

# Сегментация

In [11]:
import torch
import torchvision
from torchvision.models.segmentation import DeepLabV3_ResNet50_Weights
from torchvision.transforms.functional import to_pil_image
import cv2
import numpy as np
from PIL import Image
import time

# Задание 1

Реализуйте семантическую сегментацию u-net на видеоролике (лучше подберите так, чтоб на видео объекты двигались медленно)

In [12]:
# --- Конфигурация ---
INPUT_VIDEO_PATH = "IMG_1587.mp4"
OUTPUT_VIDEO_PATH = "output_segmented_video.mp4"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [13]:
# --- Загрузка модели ---
weights = DeepLabV3_ResNet50_Weights.DEFAULT
model = torchvision.models.segmentation.deeplabv3_resnet50(weights=weights)
model.to(DEVICE)
model.eval()

DeepLabV3(
  (backbone): IntermediateLayerGetter(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Se

In [14]:
# Получение трансформаций, рекомендованных для модели
preprocess = weights.transforms()

# Используем простую палитру для примера
palette = torch.tensor([2 ** 25 - 1, 2 ** 15 - 1, 2 ** 21 - 1])
colors = torch.as_tensor([i for i in range(21)])[:, None] * palette
colors = (colors % 255).numpy().astype("uint8")
# Добавим немного прозрачности для фона (класс 0)
colors[0, :] = 0 # Черный цвет для фона, но мы сделаем его прозрачным при наложении

In [15]:
# --- Функция для декодирования маски сегментации в цветное изображение ---
def decode_segmap(image_tensor):
    rgb_mask = Image.fromarray(image_tensor.byte().cpu().numpy()).convert("P")
    rgb_mask.putpalette(colors)
    # Конвертируем в RGB для дальнейшей обработки с OpenCV
    rgb_mask = rgb_mask.convert('RGB')
    return np.array(rgb_mask)

# --- Обработка видео ---
cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
if not cap.isOpened():
    print(f"Ошибка: {INPUT_VIDEO_PATH}")
    exit()

In [16]:
# Получаем свойства видео
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

In [20]:
# Создаем объект для записи видео
# Используем кодек 'mp4v' для MP4 файлов
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))

print(f"Video properties: {frame_width}x{frame_height} @ {fps:.2f} FPS, Total Frames: {total_frames}")

Video properties: 1080x1920 @ 59.94 FPS, Total Frames: 440


In [18]:
frame_count = 0
start_time = time.time()

while True:
    ret, frame_bgr = cap.read()
    if not ret:
        break # Конец видео

    # 1. Предобработка кадра
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(frame_rgb)
    input_tensor = preprocess(pil_image)
    input_batch = input_tensor.unsqueeze(0).to(DEVICE) # Добавляем batch dimension и отправляем на устройство

    # 2. Получение предсказания от модели
    with torch.no_grad():
        output = model(input_batch)['out'][0] # Получаем вывод и убираем batch dimension

    # 3. Постобработка: получение маски классов и декодирование в цвет
    output_predictions = output.argmax(0) # Получаем индекс класса для каждого пикселя
    mask_rgb = decode_segmap(output_predictions)

    # 4. Изменение размера маски до оригинального размера кадра
    # Важно использовать INTER_NEAREST для масок, чтобы не смешивать классы
    mask_resized_rgb = cv2.resize(mask_rgb, (frame_width, frame_height), interpolation=cv2.INTER_NEAREST)

    # Конвертируем цветную маску в BGR для смешивания с оригинальным кадром OpenCV
    mask_resized_bgr = cv2.cvtColor(mask_resized_rgb, cv2.COLOR_RGB2BGR)

    # 5. Наложение маски на оригинальный кадр
    # Создаем маску прозрачности: где класс не фон (0), там непрозрачно
    alpha_mask = (output_predictions.cpu().numpy() > 0).astype(np.uint8) * 255
    alpha_mask_resized = cv2.resize(alpha_mask, (frame_width, frame_height), interpolation=cv2.INTER_NEAREST)
    alpha_mask_resized_3ch = cv2.cvtColor(alpha_mask_resized, cv2.COLOR_GRAY2BGR) # Делаем 3-канальной

    # Применяем маску только к цветной сегментации
    masked_segmentation = cv2.bitwise_and(mask_resized_bgr, alpha_mask_resized_3ch)

    # Инвертируем маску прозрачности для оригинального кадра
    alpha_mask_inv = cv2.bitwise_not(alpha_mask_resized_3ch)
    masked_original = cv2.bitwise_and(frame_bgr, alpha_mask_inv)

    # Комбинируем оригинальный кадр (где фон) и сегментацию (где объекты)
    alpha = 0.6 # Коэффициент прозрачности для наложения
    beta = 1.0 - alpha
    output_frame = cv2.addWeighted(frame_bgr, alpha, mask_resized_bgr, beta, 0.0)


    # 6. Запись обработанного кадра
    out.write(output_frame)

    frame_count += 1
    if frame_count % 20 == 0: # Выводим прогресс каждые 20 кадров
        elapsed_time = time.time() - start_time
        avg_fps = frame_count / elapsed_time
        print(f"Processed frame {frame_count}/{total_frames} ({avg_fps:.2f} FPS)")

Processed frame 20/440 (3.22 FPS)
Processed frame 40/440 (3.22 FPS)
Processed frame 60/440 (3.23 FPS)
Processed frame 80/440 (3.22 FPS)
Processed frame 100/440 (3.25 FPS)
Processed frame 120/440 (3.24 FPS)
Processed frame 140/440 (3.26 FPS)
Processed frame 160/440 (3.24 FPS)
Processed frame 180/440 (3.26 FPS)
Processed frame 200/440 (3.23 FPS)
Processed frame 220/440 (3.25 FPS)
Processed frame 240/440 (3.25 FPS)
Processed frame 260/440 (3.26 FPS)
Processed frame 280/440 (3.26 FPS)
Processed frame 300/440 (3.26 FPS)
Processed frame 320/440 (3.25 FPS)
Processed frame 340/440 (3.25 FPS)
Processed frame 360/440 (3.26 FPS)
Processed frame 380/440 (3.26 FPS)
Processed frame 400/440 (3.26 FPS)
Processed frame 420/440 (3.26 FPS)
Processed frame 440/440 (3.27 FPS)


In [19]:
end_time = time.time()
total_time = end_time - start_time
print("-" * 30)
print(f"Video processing finished.")
print(f"Total time: {total_time:.2f} seconds")
if total_frames > 0 and total_time > 0:
     print(f"Average processing speed: {total_frames / total_time:.2f} FPS")
print(f"Output video saved to: {OUTPUT_VIDEO_PATH}")

------------------------------
Video processing finished.
Total time: 147.37 seconds
Average processing speed: 2.99 FPS
Output video saved to: output_segmented_video.mp4


# Задание 2

Реализуйте семантическую сегментацию с помощью предобученной yolo на видеоролике

In [22]:
pip install ultralytics opencv-python torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 851.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 103.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Unins

In [9]:
from ultralytics import YOLO
import cv2
import time

In [2]:
# --- Конфигурация ---
INPUT_VIDEO_PATH = "IMG_1587.mp4"
OUTPUT_VIDEO_PATH = "output_yolo_segmented_video.mp4"
# Выбираем модель YOLOv8 для сегментации ('n'-nano, 's'-small, 'm'-medium, 'l'-large, 'x'-xlarge)
# Модели побольше точнее, но медленнее. Начнем с быстрой 'n'.
MODEL_NAME = 'yolov8n-seg.pt' # '.pt' файл будет скачан автоматически при первом запуске

In [3]:
# --- Загрузка модели ---
# Модель автоматически выберет CUDA, если доступно
model = YOLO(MODEL_NAME)
print(f"Using device: {model.device}") # Посмотрим, какое устройство используется

Using device: cpu


In [6]:
# --- Обработка видео ---
cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
if not cap.isOpened():
    print(f"Error: Could not open video file {INPUT_VIDEO_PATH}")
    exit()

In [7]:
# Получаем свойства видео
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Создаем объект для записи видео
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))

print(f"Video properties: {frame_width}x{frame_height} @ {fps:.2f} FPS, Total Frames: {total_frames}")
print(f"Using YOLO model: {MODEL_NAME}")

Video properties: 1080x1920 @ 59.94 FPS, Total Frames: 440
Using YOLO model: yolov8n-seg.pt


In [10]:
print("Starting video processing...")

frame_count = 0
start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break # Конец видео

    # 1. Выполнение сегментации с помощью YOLOv8
    # 'frame' должен быть numpy array (BGR), что cv2.read() и возвращает
    # verbose=False отключает вывод детальной информации о детекции в консоль для каждого кадра
    results = model(frame, verbose=False)

    # 2. Получение кадра с наложенными масками
    # results[0] содержит результаты для первого (и единственного) изображения/кадра
    # Метод .plot() удобно рисует и bounding box'ы, и маски сегментации на кадре
    annotated_frame = results[0].plot()

    # 3. Запись обработанного кадра
    out.write(annotated_frame)

    frame_count += 1
    if frame_count % 20 == 0: # Выводим прогресс каждые 20 кадров
        elapsed_time = time.time() - start_time
        avg_fps = frame_count / elapsed_time
        print(f"Processed frame {frame_count}/{total_frames} ({avg_fps:.2f} FPS)")

Starting video processing...
Processed frame 20/440 (4.70 FPS)
Processed frame 40/440 (7.38 FPS)
Processed frame 60/440 (9.33 FPS)
Processed frame 80/440 (10.72 FPS)
Processed frame 100/440 (11.77 FPS)
Processed frame 120/440 (12.61 FPS)
Processed frame 140/440 (13.33 FPS)
Processed frame 160/440 (13.94 FPS)
Processed frame 180/440 (14.45 FPS)
Processed frame 200/440 (14.89 FPS)
Processed frame 220/440 (15.27 FPS)
Processed frame 240/440 (15.33 FPS)
Processed frame 260/440 (15.26 FPS)
Processed frame 280/440 (15.23 FPS)
Processed frame 300/440 (15.53 FPS)
Processed frame 320/440 (15.76 FPS)
Processed frame 340/440 (15.98 FPS)
Processed frame 360/440 (16.16 FPS)
Processed frame 380/440 (16.30 FPS)
Processed frame 400/440 (16.44 FPS)
Processed frame 420/440 (16.57 FPS)
Processed frame 440/440 (16.70 FPS)


In [11]:
end_time = time.time()
total_time = end_time - start_time
print("-" * 30)
print(f"Video processing finished.")
print(f"Total time: {total_time:.2f} seconds")
if total_frames > 0 and total_time > 0:
     print(f"Average processing speed: {total_frames / total_time:.2f} FPS")
print(f"Output video saved to: {OUTPUT_VIDEO_PATH}")

------------------------------
Video processing finished.
Total time: 26.36 seconds
Average processing speed: 16.69 FPS
Output video saved to: output_yolo_segmented_video.mp4
